In [ ]:
from bigmodule import M, I

# <aistudiograph>

# @param(id="m3", name="initialize")
# 交易引擎：初始化函数，只执行一次
def m3_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder
    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    

# @param(id="m3", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m3_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# @param(id="m3", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m3_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m3", name="handle_data")
def m3_handle_data_bigquant_run(context, data):
    import pandas as pd

    # 下一个交易日不是调仓日，则不生成信号
    if not context.rebalance_period.is_signal_date(data.current_dt.date()):
        return

    # 从传入的数据 context.data 中读取今天的信号数据
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    # 无当天数据返回
    if len(today_df)==0:
        return
    
    # 账户持仓
    positions = context.get_account_positions()
    
    # 持股数量
    for stock in context.instruments:
        if stock in positions.keys():
            hold_num = positions[stock].current_qty
        else:
            hold_num = 0 
        
            
        # 交易信号
        try:
            entry_condi  = today_df[today_df['instrument'] == stock].entry_condi.values[0]
            exit_condi  = today_df[today_df['instrument'] == stock].exit_condi.values[0]
        except :
            continue
        
        weight = 1 / len(context.instruments)  
        
        # 满足开仓条件
        if entry_condi:
            # 如果没有仓位
            if hold_num == 0:
                context.order_target_percent(stock, weight)
            # 如果持有仓位 
            elif hold_num > 0:
                pass 

        # 满仓平仓条件
        if exit_condi: 
            # 如果没有仓位
            if hold_num == 0:
                pass 
            # 如果有仓位
            elif hold_num > 0:
                context.order_target_percent(stock, 0)

# @param(id="m3", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m3_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m3", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m3_handle_order_bigquant_run(context, order):
    pass

# @param(id="m3", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m3_after_trading_bigquant_run(context, data):
    pass

# @module(position="-243,-1187", comment="""""", comment_collapsed=True)
m1 = M.input_features_dai.v30(
    mode="""表达式""",
    expr="""m_avg(close,5) as short_ma
m_avg(close,80) as long_ma

-- 进场条件
m_lag(short_ma, 1) <= m_lag(long_ma, 1) and short_ma > long_ma  as entry_condi 
-- 出场条件 
m_lag(short_ma, 1) >= m_lag(long_ma, 1) and short_ma < long_ma as exit_condi 
 """,
    expr_filters="""instrument in ('000002.SZ', '600519.SH')""",
    expr_tables="""cn_stock_prefactors_community""",
    extra_fields="""date,instrument""",
    order_by="""date, instrument""",
    expr_drop_na=False,
    sql="""-- 使用DAI SQL获取数据, 构建因子等, 如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B
-- 使用数据输入1/2/3里的字段: e.g. input_1.close, input_1.* EXCLUDE(date, instrument)

SELECT
    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    m_lag(close, 90) / close AS return_90,
    m_lag(close, 30) / close AS return_30,
    -- 下划线开始命名的列是中间变量, 不会在最终结果输出 (e.g. _rank_return_90)
    c_pct_rank(-return_90) AS _rank_return_90,
    c_pct_rank(return_30) AS _rank_return_30,

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_bar1d https://bigquant.com/data/datasources/cn_stock_bar1d
    cn_stock_prefactors
    -- SQL 模式不会自动join输入数据源, 可以根据需要自由灵活的使用
    -- JOIN input_1 USING(date, instrument)
WHERE
    -- WHERE 过滤, 在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤, 在窗口等计算算子之后执行, 比如 m_lag(close, 3) AS close_3, 对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
    -- _rank_return_90 是窗口函数结果，需要放在 QUALIFY 里
    AND _rank_return_90 > 0.1
    AND _rank_return_30 < 0.1
-- 按日期和股票代码排序, 从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m1"""
)

# @module(position="-242,-1099", comment="""""", comment_collapsed=True)
m4 = M.extract_data_dai.v17(
    sql=m1.data,
    start_date="""2015-01-01""",
    start_date_bound_to_trading_date=False,
    end_date="""2024-08-26""",
    end_date_bound_to_trading_date=False,
    before_start_days=60,
    debug=False,
    m_name="""m4"""
)

# @module(position="-266,-1010", comment="""交易，日线，设置初始化函数和K线处理函数，以及初始资金、基准等""", comment_collapsed=True)
m3 = M.bigtrader.v34(
    data=m4.data,
    start_date="""""",
    end_date="""""",
    initialize=m3_initialize_bigquant_run,
    before_trading_start=m3_before_trading_start_bigquant_run,
    handle_tick=m3_handle_tick_bigquant_run,
    handle_data=m3_handle_data_bigquant_run,
    handle_trade=m3_handle_trade_bigquant_run,
    handle_order=m3_handle_order_bigquant_run,
    after_trading=m3_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="""daily""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""1""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""open""",
    benchmark="""沪深300指数""",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="""m3"""
)
# </aistudiograph>

[2024-11-27 17:12:28] [info     ] input_features_dai.v30 开始运行 ..
[2024-11-27 17:12:28] [info     ] expr mode
[2024-11-27 17:12:30] [info     ] input_features_dai.v30 运行完成 [1.271s].
[2024-11-27 17:12:30] [info     ] extract_data_dai.v17 开始运行 ..
[2024-11-27 17:12:32] [info     ] data extracted: (4694, 6)
[2024-11-27 17:12:32] [info     ] extract_data_dai.v17 运行完成 [2.665s].
[2024-11-27 17:12:32] [info     ] bigtrader.v34 开始运行 ..
[2024-11-27 17:12:33] [info     ] got metadata extra from input datasource
[2024-11-27 17:12:33] [info     ] read input 'data' ..
[2024-11-27 17:12:33] [info     ] 2015-01-01, 2024-08-26, , equity, instruments=2
[2024-11-27 17:12:34] [info     ] bigtrader module V2.2.0
[2024-11-27 17:12:34] [info     ] bigtrader engine v1.10.10 2024-11-21
[2024-11-27 17:12:52] [info     ] backtest done, raw_perf_ds:dai.DataSource("_d9642d9c92714a52b9278131c5f723b6")


[2024-11-27 17:12:57] [info     ] bigtrader.v34 运行完成 [24.553s].
